In [1]:
def run_pipeline():
    # import modules
    import apache_beam as beam
    import logging
    from apache_beam.options.pipeline_options import PipelineOptions
    from datetime import datetime

    # define pipeline options
    beam_options = PipelineOptions(
        runner="DataflowRunner",
        project="caramel-medley-435805-c7",
        region="us-central1",
        job_name="ecommerce-transaction-pipeline",
        temp_location="gs://dummy-bucket-gcs-test/temppp/",
        staging_location="gs://dummy-bucket-gcs-test/stagingg/"
    )

    # variables 
    csv_path = "gs://dummy-bucket-gcs-test/transactions.csv"
    table_id = "caramel-medley-435805-c7.test_db.transactions_valid"
    BQ_SCHEMA = "transaction_id:STRING,user_id:INT64,product_id:STRING,quantity:INT64,price:FLOAT64,timestamp:TIMESTAMP,total_sales:FLOAT64"

    class ValidateAndTransform(beam.DoFn):
        def process(self, element):
            fields = element.split(",")

            try:
                # extract fields
                transaction_id = fields[0]
                user_id = fields[1]
                product_id = fields[2]
                quantity = int(fields[3])
                price = float(fields[4])
                timestamp = fields[5]

                # validate fields
                if not user_id or not product_id:
                    raise ValueError("Missing user_id or product_id")
                if quantity <= 0:
                    raise ValueError("Invalid quantity: must be greater than 0")
                if price <= 0:
                    raise ValueError("Invalid price: must be greater than 0")

                # calculate total sales
                total_sales = quantity * price

                # return as clean dict
                yield {
                    "transaction_id": transaction_id,
                    "user_id": user_id,
                    "product_id": product_id,
                    "quantity": quantity,
                    "price": price,
                    "timestamp": timestamp,
                    "total_sales": total_sales
                }
            except Exception as e:
                logging.error(f"Error processing transaction: {str(e)}")

                # Sending invalid record to side output
                yield beam.pvalue.TaggedOutput('invalid_records', element)
            
    with beam.Pipeline(options=beam_options) as pipeline:
        # Separate valid and invalid transactions
        valid_transactions, invalid_transactions = (
            pipeline 
            | "read from source" >> beam.io.ReadFromText(csv_path, skip_header_lines=1)
            | "validate and transform" >> beam.ParDo(ValidateAndTransform()).with_outputs("invalid_records", main="valid_records")
        )

        # Write valid records to BigQuery
        valid_transactions | "write to bq" >> beam.io.WriteToBigQuery(
            table=table_id,
            schema=BQ_SCHEMA,
            write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
            create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
            custom_gcs_temp_location="gs://dummy-bucket-gcs-test/stage/"
        )

        # Write invalid records to GCS bucket as JSON
        invalid_transactions | "write to gcs" >> beam.io.WriteToText(
            "gs://dummy-bucket-gcs-test/invalid/invalid_transactions",
            file_name_suffix=".json"
        )

        # Set logging level to INFO
        logging.getLogger().setLevel(logging.INFO)

if __name__ == "__main__":
    run_pipeline()


INFO:apache_beam.runners.dataflow.dataflow_runner:Pipeline has additional dependencies to be installed in SDK worker container, consider using the SDK container image pre-building workflow to avoid repetitive installations. Learn more on https://cloud.google.com/dataflow/docs/guides/using-custom-containers#prebuild
INFO:root:Using provided Python SDK container image: gcr.io/cloud-dataflow/v1beta3/beam_python3.10_sdk:2.59.0
INFO:root:Python SDK container image set to "gcr.io/cloud-dataflow/v1beta3/beam_python3.10_sdk:2.59.0" for Docker environment
INFO:apache_beam.runners.dataflow.internal.apiclient:Starting GCS upload to gs://dummy-bucket-gcs-test/stagingg/ecommerce-transaction-pipeline.1729248094.534198/submission_environment_dependencies.txt...
INFO:apache_beam.runners.dataflow.internal.apiclient:Completed GCS upload to gs://dummy-bucket-gcs-test/stagingg/ecommerce-transaction-pipeline.1729248094.534198/submission_environment_dependencies.txt in 3 seconds.
INFO:apache_beam.runners.da